In [1]:
import pandas as pd 

In [2]:
import re

pattern = re.compile(
    r'^(\w{3}\s+\d+\s+\d+:\d+:\d+)\s+'  # date
    r'\S+\s+kernel:.*?\s+'               # host + kernel
    r'(PERMIT|DENY)\s+'                  # action
    r'FW=\S+\s+'                         # fw (à ignorer)
    r'RULE=(\S+)\s+'                     # rule_id
    r'IN=(\S*)\s+'                       # interface_in
    r'OUT=(\S*)\s+'                      # interface_out
    r'.*?'                               # MAC etc.
    r'SRC=(\S+)\s+'                      # ip_src
    r'DST=(\S+)\s+'                      # ip_dst
    r'.*?'                               # LEN TOS etc.
    r'PROTO=(\S+)\s+'                    # protocol
    r'SPT=(\d+)\s+'                      # port_src
    r'DPT=(\d+)'                         # port_dst
)

def parse_line(line):
    m = pattern.search(line)
    if m:
        return m.groups()
    return None

# test sur une ligne
sample = 'Mar 20 01:29:24 159.84.146.99 kernel: [54783294.108218] DENY FW=6 RULE=999 IN=eth0 OUT= MAC=c4:37:72:f9:da:ed:fe:37:72:f9:da:ed:08:00 SRC=94.102.61.47 DST=159.84.146.99 LEN=44 TOS=0x00 PREC=0x00 TTL=238 ID=54321 PROTO=TCP SPT=52502 DPT=3178 WINDOW=65535 RES=0x00 SYN URGP=0'
print(parse_line(sample))

('Mar 20 01:29:24', 'DENY', '999', 'eth0', '', '94.102.61.47', '159.84.146.99', 'TCP', '52502', '3178')


In [3]:
cols = ["datetime", "action", "rule_id", "interface_in", "interface_out", "ip_src", "ip_dst", "protocol", "port_src", "port_dst"]

with open("../data_logs/log_brute/log_brute.log", "r") as f:
    rows = [parse_line(line) for line in f]

df = pd.DataFrame([r for r in rows if r is not None], columns=cols)
print(df.shape)
print(df.head(3))

(11997646, 10)
          datetime  action rule_id interface_in interface_out          ip_src  \
0  Mar 20 01:29:24    DENY     999         eth0                  94.102.61.47   
1  Mar 20 01:29:25    DENY     999         eth0                176.111.174.85   
2  Mar 20 01:29:27  PERMIT       1         eth0                 66.249.65.106   

          ip_dst protocol port_src port_dst  
0  159.84.146.99      TCP    52502     3178  
1  159.84.146.99      TCP    48739     2231  
2  159.84.146.99      TCP    50501      443  


In [4]:
df["datetime"] = pd.to_datetime(df["datetime"], format="%b %d %H:%M:%S")


df["datetime"] = df["datetime"].apply(
    lambda x: x.replace(year=2026) if x.month <= 2 else x.replace(year=2025)
)

df_filtered = df[(df["datetime"] >= "2025-11-01") & (df["datetime"] < "2026-03-01")]
print(df_filtered.shape)
print(df_filtered["datetime"].dt.to_period("M").value_counts().sort_index())

(4572903, 10)
datetime
2025-11    1212335
2025-12    1620630
2026-01    1250863
2026-02     489075
Freq: M, Name: count, dtype: int64


In [5]:
df_filtered.to_parquet("../data_logs/log_brute/log_clean.parquet", index=False)